In [16]:
# Import the function that creates all database tables
from database import create_tables

# Execute the function to create the tables defined by the SQLAlchemy models
create_tables()

2026-04-03 08:58:56,402 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 08:58:56,404 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("pipe")
2026-04-03 08:58:56,404 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 08:58:56,406 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("weather_station")
2026-04-03 08:58:56,406 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 08:58:56,408 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("rainfall")
2026-04-03 08:58:56,408 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 08:58:56,410 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("air_humidity")
2026-04-03 08:58:56,412 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 08:58:56,412 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("air_temperature")
2026-04-03 08:58:56,414 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 08:58:56,414 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("pipe_seismic_impact")
2026-04-03 08:58:56,414 INFO sqlalc

In [17]:
# Import the session factory configured in the database module
from database import SessionLocal

# Create a new SQLAlchemy session to interact with the database
session = SessionLocal()

In [18]:
import pandas as pd
from sqlalchemy.orm import Session

# ---------------------------------------------------------------------------
# 1. Read Excel file into a DataFrame
# ---------------------------------------------------------------------------

def read_excel_to_dataframe(
    excel_path: str,
    sheet_name: int | str = 0,
    drop_empty_rows: bool = True,
) -> pd.DataFrame:
    """
    Read an Excel file into a pandas DataFrame.

    Parameters
    ----------
    excel_path : str
        Path to the Excel file on disk.
    sheet_name : int or str, optional
        Sheet index (0-based) or sheet name. Default is 0 (first sheet).
    drop_empty_rows : bool, optional
        If True, rows that are completely empty will be removed.

    Returns
    -------
    df : pandas.DataFrame
        DataFrame containing the data from the selected sheet.
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    if drop_empty_rows:
        df = df.dropna(how="all")

    return df

In [19]:
# ---------------------------------------------------------------------------
# 2. Upload a DataFrame to a database table (generic)
# ---------------------------------------------------------------------------

def upload_dataframe_to_table(
    df: pd.DataFrame,
    model_cls,
    engine,
):
    """
    Upload a pandas DataFrame to a database table using SQLAlchemy.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the data to be inserted.
        Column names should match the attributes of `model_cls`.
    model_cls : Base subclass
        SQLAlchemy ORM model class corresponding to the target table.
    engine : sqlalchemy.Engine
        SQLAlchemy engine connected to the target database.

    Notes
    -----
    - This function:
        * keeps only the columns that exist in the model's table,
        * replaces pandas NA values with Python None,
        * uses bulk_insert_mappings for efficient insertion.
    - It assumes the table is already created in the database.
    """

    # 1. Keep only valid columns
    model_columns = {col.name for col in model_cls.__table__.columns}
    valid_cols = [c for c in df.columns if c in model_columns]
    df_clean = df[valid_cols].copy()

    # -------------------------------------------------------------------
    # 2. Handle DATE columns automatically
    # -------------------------------------------------------------------
    for col in model_cls.__table__.columns:
        if str(col.type) == "DATE" and col.name in df_clean.columns:
            df_clean[col.name] = pd.to_datetime(df_clean[col.name], errors="coerce").dt.date

    # -------------------------------------------------------------------
    # 3. Replace NaN with None
    # -------------------------------------------------------------------
    records = (
        df_clean
        .where(pd.notna(df_clean), None)
        .to_dict(orient="records")
    )

    # -------------------------------------------------------------------
    # 4. Insert into DB
    # -------------------------------------------------------------------
    with Session(engine) as session:
        session.bulk_insert_mappings(model_cls, records)
        session.commit()


In [20]:
# ---------------------------------------------------------------------------
# 3. Example usage (to be adapted in your own scripts / notebooks)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    """
    Example workflow to demonstrate how to load data from an Excel file
    into the database.

    This example uses a sample file named 'EXAMPLE_DATA_DB_FINAL.xlsx', which is
    included in the repository. Users can replace this file with their
    own dataset, provided that the column names match the database schema.
    """

    # Import database engine and ORM models
    from database import engine
    from schema import Pipe, HydraulicProperties, Inspection, Defect # import other models as needed

    # -----------------------------------------------------------------------
    # Step 1: Read data from Excel
    # -----------------------------------------------------------------------
    excel_file = "EXAMPLE_DATA_DataBase.xlsx"   # Example file provided in the repo

    pipes_df = read_excel_to_dataframe(
        excel_path=excel_file,
        sheet_name="PIPES"
    )

    hydraulic_df= read_excel_to_dataframe(excel_path=excel_file,
        sheet_name="HYDRAULIC_PROPERTIES" )

    CCTV_df= read_excel_to_dataframe(excel_path=excel_file,
        sheet_name="CCTV" )

    Defects_df= read_excel_to_dataframe(excel_path=excel_file,
        sheet_name="DEFECTS" )

    # -----------------------------------------------------------------------
    # Step 2: Upload data to database
    # -----------------------------------------------------------------------
    upload_dataframe_to_table(
        df=pipes_df,
        model_cls=Pipe,
        engine=engine,
    )
    upload_dataframe_to_table(
        df=hydraulic_df,
        model_cls=HydraulicProperties,
        engine=engine,
    )
    upload_dataframe_to_table(
        df=CCTV_df,
        model_cls=Inspection,
        engine=engine,
    )
    upload_dataframe_to_table(
        df=Defects_df,
        model_cls=Defect,
        engine=engine,
    )

    print("Data successfully uploaded to the database.")

2026-04-03 08:59:00,719 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 08:59:00,719 INFO sqlalchemy.engine.Engine INSERT INTO pipe ("Pipe_ID", "Installation_year", "Diameter", "Material", "Pipe_length", "Depth", "UP_depth", "DW_depth", "Sewage_type", "Groundwater_level", "Land_use", "Land_cover", "Sewer_category", "Soil_type", "Road_above") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
2026-04-03 08:59:00,719 INFO sqlalchemy.engine.Engine [cached since 278.8s ago] [(1000, 1919, 160, 'PE', 134.25, 17.37, 17.83, 16.92, 'Local', 9.03, 'Road area', 'Urban', 'WI', 'Silt and clay', 0), (1001, 1988, 470, 'CONC', 252.07, 24.06, 16.15, 31.98, 'Local', 2.85, 'Open space', 'Urban', 'Wastewater', 'Unknown', 1), (1002, 1999, 180, 'UNDEF', 42.21, 6.26, 5.07, 7.44, 'Local', 9.54, 'Business', 'Vegetation', 'Wastewater', 'Boulders to massive', 0), (1003, 1996, 750, 'CI', 5.73, nan, 1.28, nan, 'Transmission', 30.16, 'Open space', 'Vegetation', 'Waste', 'Unknown', 0), (1004, 1921, 180, 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [21]:
# Code if you want to drop the information from each entity and create a new one.

# Pipe.__table__.drop(engine)
# Pipe.__table__.create(engine)
# HydraulicProperties.__table__.drop(engine)
# HydraulicProperties.__table__.create(engine)
# Defect.__table__.drop(engine)
# Defect.__table__.create(engine)
# Inspection.__table__.drop(engine)
# Inspection.__table__.create(engine)

2026-04-03 09:00:18,550 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 09:00:18,551 INFO sqlalchemy.engine.Engine 
DROP TABLE pipe
2026-04-03 09:00:18,551 INFO sqlalchemy.engine.Engine [no key 0.00037s] ()
2026-04-03 09:00:18,567 INFO sqlalchemy.engine.Engine COMMIT
2026-04-03 09:00:18,568 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 09:00:18,570 INFO sqlalchemy.engine.Engine 
CREATE TABLE pipe (
	"Pipe_ID" INTEGER NOT NULL, 
	"Installation_year" INTEGER, 
	"Diameter" INTEGER, 
	"Material" VARCHAR(20), 
	"Pipe_length" FLOAT, 
	"Depth" FLOAT, 
	"UP_depth" FLOAT, 
	"DW_depth" FLOAT, 
	"Slope" FLOAT, 
	"Sewage_type" VARCHAR, 
	"Groundwater_level" FLOAT, 
	"Land_use" VARCHAR, 
	"Land_cover" VARCHAR, 
	"Trees_nearby" INTEGER, 
	"Shape" VARCHAR, 
	"Bedding" VARCHAR, 
	"Joint_type" VARCHAR, 
	"Population" INTEGER, 
	"Climatic_condition" FLOAT, 
	"Sewer_category" VARCHAR, 
	"Sewer_connections" INTEGER, 
	"Weather_station_ID" INTEGER, 
	"Water_quality" FLOAT, 
	"Backfill